<a href="https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I'm working on predicting the order in which items should be presented to a user, based on their potential relevance or engagement. Therefore, this is a **ranking** task.

Here's why:
*   **Goal:** The primary goal is to sort a list of available items (e.g., products, search results, recommendations) in a way that maximizes a specific objective, such as user clicks, purchases, or satisfaction.
*   **Output:** The output is an ordered list of items, not a binary classification (like spam/not spam), a grouping (clustering), or a single numerical score that doesn't imply order relative to other items.
*   **Evaluation:** Ranking models are typically evaluated using metrics like Normalized Discounted Cumulative Gain (NDCG), Mean Average Precision (MAP), or Kendall's Tau, which specifically measure the quality of an ordered list.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For a ranking task, we would predict the **relevance score** or **propensity score** of an item for a given user in a particular context.

This label typically comes from an **observed outcome** that serves as a proxy for relevance. Examples include:

*   **Click-Through Rate (CTR):** If a user clicks on an item, it's considered more relevant. The label could be binary (clicked/not clicked) or a smoothed version.
*   **Conversion Rate (e.g., Purchase):** For e-commerce, a purchase indicates a strong relevance. This is often a stronger signal than a click.
*   **Time Spent:** For content recommendations, more time spent viewing an item suggests higher engagement and relevance.
*   **Explicit Feedback:** User ratings (e.g., 1-5 stars, likes/dislikes) provide direct feedback on relevance.

In many real-world ranking systems, these observed outcomes are combined or transformed into a continuous relevance score. For instance, a click might contribute 0.1 to the score, a purchase 1.0, and a long view duration proportionally. This approach uses observed user behavior as a **defined rule** to create a proxy target variable.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

For ranking tasks, a highly defensible and widely used metric is **Normalized Discounted Cumulative Gain (NDCG)**.

### Why NDCG?

1.  **Positional Importance:** NDCG inherently values highly relevant items appearing higher in the ranked list more than those appearing lower. A relevant item at position 1 contributes more to the score than the same relevant item at position 10.
2.  **Graded Relevance:** Unlike binary metrics (like precision@k), NDCG can incorporate graded relevance levels (e.g., highly relevant, somewhat relevant, irrelevant), which aligns well with a continuous relevance score target.
3.  **Normalization:** It's normalized to a value between 0 and 1, making it comparable across different queries or datasets. A perfect ranking (all most relevant items at the top) results in an NDCG of 1.

### What number means 'good'?

What constitutes a 'good' NDCG score is highly context-dependent and varies significantly based on the domain, dataset, and business goals. However, general interpretations are:

*   **NDCG of 1.0:** Represents a perfect ranking, where the model's output exactly matches the ideal ranking.
*   **NDCG > 0.8:** Generally considered excellent performance, indicating that the model is performing very well at surfacing highly relevant items at the top.
*   **NDCG between 0.6 and 0.8:** Often considered good performance, suggesting a strong ability to rank, but with room for improvement.
*   **NDCG < 0.5:** Usually indicates poor performance, where the model is not effectively ranking items, or it's performing only slightly better than random.

**For our specific task**, an **NDCG of 0.75 or higher** would be considered a 'good' initial target. This value would signify that our ranking model is successfully identifying and elevating relevant items to the top positions, leading to a significantly better user experience than a random or rule-based approach. The ultimate 'good' threshold would be set in collaboration with stakeholders, considering the baseline performance of existing systems (if any) and business impact.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

For a ranking model, the fundamental unit of analysis, which forms one row in our training dataframe, is a **(User, Item, Context, Target)** interaction. More precisely, it represents a candidate item being considered for a specific user in a given context, along with its associated relevance target.

Each row in the dataframe will represent:

*   **A unique user:** Identified by a `user_id`.
*   **A candidate item:** Identified by an `item_id`.
*   **Contextual information:** Features describing the situation in which the user might encounter the item (e.g., `timestamp`, `device_type`, `search_query`).
*   **User features:** Characteristics of the user (e.g., `user_age`, `user_gender`, `user_location`, `user_history_embedding`).
*   **Item features:** Characteristics of the item (e.g., `item_category`, `item_price`, `item_description_embedding`, `item_popularity`).
*   **Interaction features:** Features derived from the interaction between the user and item, or their history (e.g., `time_since_last_interaction`, `user_viewed_item_category_count`).
*   **Target Relevance:** The observed or engineered relevance score for this specific (user, item, context) pair, which the model will learn to predict (e.g., `relevance_score`).

So, **one row = one (user, item, context) candidate pair with its observed relevance.**

In [1]:
import pandas as pd
import numpy as np

# Create a hypothetical dataframe representing the unit of analysis
data = {
    'user_id': [1, 1, 1, 2, 2, 3, 3, 3, 3],
    'item_id': [101, 102, 103, 101, 104, 102, 103, 104, 105],
    'timestamp': pd.to_datetime([
        '2023-01-01 10:00:00', '2023-01-01 10:05:00', '2023-01-01 10:10:00',
        '2023-01-02 11:00:00', '2023-01-02 11:02:00',
        '2023-01-03 09:00:00', '2023-01-03 09:05:00', '2023-01-03 09:10:00', '2023-01-03 09:15:00'
    ]),
    'device_type': ['mobile', 'mobile', 'desktop', 'mobile', 'desktop', 'mobile', 'mobile', 'desktop', 'mobile'],
    'user_age': [25, 25, 25, 30, 30, 40, 40, 40, 40],
    'item_category': ['electronics', 'books', 'home_goods', 'electronics', 'apparel', 'books', 'home_goods', 'apparel', 'electronics'],
    'item_price': [500.0, 15.0, 45.0, 520.0, 30.0, 16.0, 48.0, 32.0, 120.0],
    'relevance_score': [0.8, 0.2, 0.5, 0.9, 0.1, 0.7, 0.3, 0.6, 0.4] # This is our target variable
}

df_unit_of_analysis = pd.DataFrame(data)

print("Sample of the unit of analysis dataframe (one row = one user-item-context candidate pair with relevance score):")
print(df_unit_of_analysis.head())

print("\nDataFrame Info:")
df_unit_of_analysis.info()

Sample of the unit of analysis dataframe (one row = one user-item-context candidate pair with relevance score):
   user_id  item_id           timestamp device_type  user_age item_category  \
0        1      101 2023-01-01 10:00:00      mobile        25   electronics   
1        1      102 2023-01-01 10:05:00      mobile        25         books   
2        1      103 2023-01-01 10:10:00     desktop        25    home_goods   
3        2      101 2023-01-02 11:00:00      mobile        30   electronics   
4        2      104 2023-01-02 11:02:00     desktop        30       apparel   

   item_price  relevance_score  
0       500.0              0.8  
1        15.0              0.2  
2        45.0              0.5  
3       520.0              0.9  
4        30.0              0.1  

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----  

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

For a ranking task, machine learning often outperforms fixed rules because the underlying patterns of user relevance are inherently **complex, dynamic, and non-linear**, making them "too messy for an if-statement."

Here's why ML beats a fixed rule:

1.  **High Dimensionality and Interdependencies:** User preferences and item characteristics are not isolated. For example, a user might like action movies (`item_category`), but only if they are new (`timestamp` related) and have high ratings (`item_popularity`). A fixed rule would struggle to capture the intricate interplay between `user_age`, `item_price`, `device_type`, `item_category`, and `relevance_score` simultaneously.

2.  **Non-linear Relationships:** The impact of a feature on relevance is rarely linear. A user might prefer items within a certain price range, but items that are too cheap or too expensive might both be less relevant. `if price > X and price < Y` is too simplistic when there are complex, non-linear interactions between price, user demographics, and category.

3.  **Context-Dependence:** Relevance is highly contextual. An item might be relevant to a user searching for work tools on a desktop during business hours but irrelevant to the same user browsing for entertainment on mobile in the evening. Fixed rules would require an explosion of `if-else` conditions to cover all possible contexts.

4.  **Evolving Preferences:** User tastes and item popularity change over time. A fixed rule system would require constant manual updates to adapt to new trends, seasonal changes, or emerging items. ML models, especially those retrained periodically, can learn and adapt to these evolving patterns automatically.

5.  **Implicit Feedback and Latent Factors:** User interactions (clicks, views, purchases) provide implicit feedback that often hides complex latent factors (e.g., a user's *mood*, or an item's *quality* beyond its listed attributes). ML models can uncover these hidden patterns and signals that are impossible to hard-code into a set of `if` statements.

6.  **Scalability:** As the number of users, items, and features grows, creating and maintaining a rule-based system becomes unmanageable. An ML model can scale to millions of interactions and features more effectively, learning patterns from vast datasets.

In essence, while simple rules might capture basic preferences (e.g., "show electronics to users who bought electronics"), they fail to capture the nuanced, dynamic, and often counter-intuitive interactions that drive true relevance, which ML algorithms are designed to learn from data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.